# IDO-AUM Lite: Colab GPU runner
Develop/test on Windows; run full experiments here. Select **Runtime > Change runtime type > GPU**.

This is a thin entry notebook. Research modules are planned, not yet implemented; run cells check their availability. All training, checkpointing, metrics and plotting belong in Python modules. No experiment runs automatically during setup.

Use one stable RUN_ID across sessions. Source/data under `/content` are disposable; configuration, checkpoints, histories and results belong on Drive. Never run concurrent writers for one RUN_ID. Clear notebook outputs before committing.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/Avilas01/ido-aum-lite.git'
REPO_REF = 'main'  # Prefer a reviewed commit SHA; recovery uses the saved SHA.
RUN_ID = 'run-001'  # Reuse this on restart; change for a distinct experiment.
if not RUN_ID or any(x not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for x in RUN_ID):
    raise ValueError('RUN_ID must contain only letters, digits, hyphens or underscores.')
REPO_DIR = Path('/content/ido-aum-lite')
DATA_ROOT = Path('/content/data')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/ido-aum-lite/experiments') / RUN_ID
RUNS_ROOT = EXPERIMENT_ROOT / 'runs'
RESULTS_ROOT = EXPERIMENT_ROOT / 'results'
BENCHMARK_ROOT = EXPERIMENT_ROOT / 'benchmark'
FROZEN_CONFIG = EXPERIMENT_ROOT / 'experiment_config.json'
MANIFEST_PATH = EXPERIMENT_ROOT / 'manifest.json'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
probe = EXPERIMENT_ROOT / '.write_probe'
probe.write_text('ok', encoding='utf-8')
probe.unlink()
saved_manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if FROZEN_CONFIG.exists() != MANIFEST_PATH.exists():
    raise RuntimeError('Incomplete config/manifest publication: inspect Drive artifacts before proceeding; do not overwrite them.')
TARGET_REF = saved_manifest['git_sha'] if saved_manifest else REPO_REF
print('Persistent experiment:', EXPERIMENT_ROOT)
print('Mode:', 'resume frozen experiment' if saved_manifest else 'new experiment; benchmark and freeze required')

## Source and dependencies
Set REPO_URL before continuing. On recovery the saved commit overrides REPO_REF. Existing dirty checkouts are rejected. Drive needs space for shared warm-up/final checkpoints and full optimizer resume states; check its storage quota before starting.

The notebook uses requirements-colab.txt through the CPU-safe src.colab_dependencies helper. It checks installed distribution metadata, actual imports, the torchvision torch-version requirement, CUDA availability and a tiny compiled CUDA operator in a fresh subprocess. This does not download data or train a model.

A healthy GPU-enabled pair is preserved. Missing/broken pairs are repaired **together** using the official torch 2.14.1 + torchvision 0.29.1 CUDA 12.6 wheels. If GPU runtime selection is the problem, enable a GPU before installation instead of reinstalling packages. Non-torch packages are installed only as needed, without --upgrade.

**After changing torch/torchvision, setup intentionally stops with RestartRequired. Select Runtime > Restart session (or Restart runtime), then rerun setup with the same RUN_ID.** Do not continue with cached native libraries. The next pass keeps the repaired pair. For resumed experiments, restore recorded environment versions before proceeding if they differ. Never run requirements-colab.txt on Windows. Pair source: https://pytorch.org/get-started/previous-versions/


### Private repository sign-in
Keep the repository private. Create a fine-grained GitHub personal access token with resource owner Avilas01, **Only select repositories > ido-aum-lite**, and **Repository permissions > Contents: Read-only**. Give it a short expiration. In the next cell, paste it into the hidden prompt, never into notebook source, REPO_URL, output or chat. For a public repository leave the prompt blank. Re-enter it after restarting the session when setup asks; it is not persisted. The same transient authentication covers clone and fetch.

If clone fails, the cell now shows Git's underlying error. Exit 128 alone is generic; errors can also indicate the wrong URL, network problems, or an existing non-repository folder. No folder is automatically deleted.


In [ ]:
def checkout_source():
    # This setup code must run before project modules have been cloned.
    import base64
    from getpass import getpass
    from urllib.parse import urlsplit

    parsed = urlsplit(REPO_URL)
    if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password:
        raise ValueError('Use a plain https://github.com/owner/repository.git URL, without credentials.')
    if 'YOUR-USERNAME' in REPO_URL:
        raise ValueError('Set REPO_URL to your GitHub repository before setup.')
    token = getpass('GitHub token for private repo (hidden; blank for public): ').strip()
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    # Never inherit HTTP tracing that could disclose authentication headers.
    for name in ('GIT_TRACE', 'GIT_TRACE_CURL', 'GIT_CURL_VERBOSE'):
        env.pop(name, None)
    authorization = ''
    if token:
        authorization = base64.b64encode(('x-access-token:' + token).encode()).decode()
        count = int(env.get('GIT_CONFIG_COUNT', '0'))
        env['GIT_CONFIG_COUNT'] = str(count + 1)
        env[f'GIT_CONFIG_KEY_{count}'] = f'http.{REPO_URL}.extraheader'
        env[f'GIT_CONFIG_VALUE_{count}'] = 'Authorization: Basic ' + authorization

    def git(*args, cwd=REPO_DIR):
        result = subprocess.run(['git', *map(str, args)], cwd=cwd, env=env,
                                capture_output=True, text=True, check=False)
        if result.returncode:
            message = result.stderr.strip()
            for secret in (token, authorization):
                if secret:
                    message = message.replace(secret, '[REDACTED]')
            raise RuntimeError(f'Git {args[0]} failed (exit {result.returncode}): {message}. '
                               'For a private repo, use a token with access to this repository and Contents: Read-only.')
        return result.stdout.strip()

    try:
        if not REPO_DIR.exists():
            git('clone', REPO_URL, REPO_DIR, cwd=REPO_DIR.parent)
        if not (REPO_DIR / '.git').is_dir():
            raise RuntimeError('REPO_DIR exists but is not a Git checkout. Inspect it or choose a different REPO_DIR; no files were deleted.')
        if git('status', '--porcelain'):
            raise RuntimeError('Checkout has local changes. Preserve/review them before selecting a commit.')
        if git('remote', 'get-url', 'origin') != REPO_URL:
            raise RuntimeError('Existing checkout has a different origin; use the intended repository.')
        git('fetch', 'origin')
        # Resolve a branch against the freshly fetched remote; recorded SHAs remain exact.
        remote_ref = git('rev-parse', '--verify', f'refs/remotes/origin/{TARGET_REF}') if TARGET_REF == 'main' else TARGET_REF
        git('checkout', '--detach', remote_ref)
        sha = git('rev-parse', 'HEAD')
        if saved_manifest and sha != saved_manifest['git_sha']:
            raise RuntimeError('Code identity differs from the recorded experiment.')
        return sha
    finally:
        # Authentication is transient: not in URL, Git config, disk, or global environment.
        env.clear()
        token = authorization = ''

GIT_SHA = checkout_source()
print('Checked out:', GIT_SHA)


In [ ]:
# No torch imports occur until dependency setup has passed in a fresh process.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.colab_dependencies import ensure_colab_dependencies
dependency_status = ensure_colab_dependencies(REPO_DIR)
import torch
import torchvision
import src
ENVIRONMENT = {
    'python': platform.python_version(),
    **dependency_status['versions'],
    'cuda': dependency_status['cuda'],
    'gpu': dependency_status['gpu'],
}
print(json.dumps(ENVIRONMENT, indent=2))
ACKNOWLEDGE_GPU_CHANGE = False  # Set True only for a reviewed change in assigned GPU.
if saved_manifest:
    previous = saved_manifest['environment']
    package_changes = {key: (previous.get(key), value) for key, value in ENVIRONMENT.items()
                       if key in previous and key != 'gpu' and previous.get(key) != value}
    if package_changes:
        print('Changed versions:', package_changes)
        raise RuntimeError('Restore recorded dependency/runtime versions before resuming; do not change the scientific configuration.')
    if ENVIRONMENT['gpu'] != previous['gpu']:
        if not ACKNOWLEDGE_GPU_CHANGE:
            raise RuntimeError('Colab assigned a different GPU. Set ACKNOWLEDGE_GPU_CHANGE=True to document and accept the reproducibility limitation.')
        from datetime import datetime, timezone
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        (EXPERIMENT_ROOT / f'resume_environment_{stamp}.json').write_text(
            json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')
        print('GPU change acknowledged and recorded; bitwise equivalence is not guaranteed.')
if saved_manifest:
    digest = hashlib.sha256(FROZEN_CONFIG.read_bytes()).hexdigest()
    if digest != saved_manifest['config_sha256']:
        raise RuntimeError('Frozen configuration checksum mismatch.')
config_path = FROZEN_CONFIG if saved_manifest else REPO_DIR / 'configs/default.json'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from config import load_config
config = load_config(config_path)
print('Configuration:', json.dumps(config, indent=2))
for label, path in [('source', REPO_DIR), ('data cache', DATA_ROOT), ('checkpoints/logs', RUNS_ROOT), ('results', RESULTS_ROOT)]:
    print(f'{label}: {path}')

## PyTorch and GPU check
Run after dependency setup succeeds. If setup requested a runtime restart, restart and rerun setup first. This check only prints versions and GPU availability; it downloads no data and performs no training.


In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Module command helper
The following cells require the planned research modules. They deliberately stop if those modules do not exist. Model/data loops, scoring, saving and integrity validation must be implemented in those modules, never duplicated in this notebook.

In [ ]:
def run_module(module, *args):
    module_file = REPO_DIR.joinpath(*module.split('.')).with_suffix('.py')
    if not module_file.is_file():
        raise RuntimeError(f'{module} is not implemented yet. Complete the corresponding implementation-plan stage first.')
    subprocess.run([sys.executable, '-m', module, *map(str, args)], cwd=REPO_DIR, check=True)

NOISES_TO_RUN = [0.2, 0.4]  # Use one condition per session if needed; reuse the same RUN_ID/configuration.

## Optional hosted tests
PyTorch-dependent tests run here once implemented. Windows tests never need PyTorch. This cell installs only test extras and runs the suite; it does not download data or train.


In [ ]:
RUN_COLAB_TESTS = False
if RUN_COLAB_TESTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-dev.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)


## Optional tiny GPU smoke (later implementation stages)
This entry point is ready for the planned training runner. Run it only once src.train exists. Smoke outputs are isolated from full experiments; no benchmark or frozen-run artifacts are overwritten.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run_module('src.train', '--smoke', '--device', 'cuda', '--data-root', DATA_ROOT,
               '--output-root', EXPERIMENT_ROOT / 'smoke')


## New experiments only: benchmark, then freeze
Skip these two cells on restart. Benchmark outputs are isolated from full results. Include both views, statistics/CE inference, loading and persistent checkpoint I/O. If necessary, select the plan's uniform fallback in central configuration and rebenchmark using `--config`. Never use test/detection accuracy for that choice.

The benchmark must save its candidate scientific settings at `benchmark/experiment_config.json`. Set `FREEZE_APPROVED=True` only after the runtime estimate fits the total project budget. No full experiment begins until configuration is frozen.

In [ ]:
if not FROZEN_CONFIG.exists():
    run_module('src.train', '--benchmark', '--config', REPO_DIR / 'configs/default.json',
               '--noise', 0.2, '--device', 'cuda', '--data-root', DATA_ROOT,
               '--output-root', BENCHMARK_ROOT)
else:
    print('Existing frozen experiment: benchmark skipped.')

In [ ]:
FREEZE_APPROVED = False  # New runs only: set True after reviewing benchmark/fallback.
if not FROZEN_CONFIG.exists():
    if not FREEZE_APPROVED:
        raise RuntimeError('Review the benchmark/runtime budget before freezing a new experiment.')
    candidate = BENCHMARK_ROOT / 'experiment_config.json'
    raw_config = candidate.read_bytes()
    load_config(candidate)  # Reuse portable configuration validation before publication.
    manifest = {'run_id': RUN_ID, 'git_sha': GIT_SHA,
                'config_sha256': hashlib.sha256(raw_config).hexdigest(), 'environment': ENVIRONMENT}
    config_tmp = FROZEN_CONFIG.with_suffix('.json.tmp')
    manifest_tmp = MANIFEST_PATH.with_suffix('.json.tmp')
    config_tmp.write_bytes(raw_config)
    manifest_tmp.write_text(json.dumps(manifest, indent=2) + chr(10), encoding='utf-8')
    if hashlib.sha256(config_tmp.read_bytes()).hexdigest() != manifest['config_sha256']:
        raise RuntimeError('Persistent configuration write verification failed.')
    config_tmp.replace(FROZEN_CONFIG)
    manifest_tmp.replace(MANIFEST_PATH)
    print('Published frozen configuration and manifest:', EXPERIMENT_ROOT)
else:
    print('Existing configuration preserved:', FROZEN_CONFIG)


## Run or resume
These commands are identical on first execution and restart. The runner must start missing stages, restore interrupted branches and skip complete branches, rejecting config/code/checksum mismatches. It must persist full state after each completed epoch and retain the previous verified checkpoint. It may repeat unfinished epoch work; it must not duplicate completed histories/logs.

Before running, confirm the printed paths point to Drive. Full state includes model/BatchNorm, optimizer/scaler, epoch, WE/margins/counts, all RNG/DataLoader states, configuration and dataset/noisy-label checksums.

In [ ]:
if not FROZEN_CONFIG.exists() or not MANIFEST_PATH.exists():
    raise RuntimeError('Freeze the benchmark-selected configuration and manifest before training.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required for full runs.')
for noise in NOISES_TO_RUN:
    run_module('src.train', '--config', FROZEN_CONFIG, '--noise', noise,
               '--device', 'cuda', '--data-root', DATA_ROOT,
               '--output-root', RUNS_ROOT, '--resume')

## Evaluate, plot and inspect persistent outputs
Evaluation alone owns test accuracy, detector ROC-AUC and metrics.csv. It must cache completed checkpoint identities to avoid repeated evaluation/duplicate rows. Plots use saved metrics/logs. Both stages write directly to Drive.

In [ ]:
run_module('src.evaluate', '--device', 'cuda', '--data-root', DATA_ROOT,
           '--runs', RUNS_ROOT, '--output-root', RESULTS_ROOT)
run_module('src.plots', '--results', RESULTS_ROOT, '--runs', RUNS_ROOT)
import pandas as pd
from IPython.display import display, Image
metrics_path = RESULTS_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path))
for figure in sorted((RESULTS_ROOT / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))
print('Persistent metrics:', metrics_path)
print('Persistent checkpoints/logs:', RUNS_ROOT)

## Restart checklist
1. Reopen this notebook with a GPU runtime; mount the same Drive account and reuse RUN_ID. Do not run two sessions against one ID.
2. Rerun setup/dependencies/GPU/configuration cells. If torch changes, restart the runtime when instructed and rerun setup again before continuing. Use saved Git SHA/environment. Recreate the `/content` checkout/data cache as needed; do not erase Drive outputs.
3. Skip benchmark/freeze for an existing experiment. If config/manifest or checkpoint publication is incomplete, inspect and recover the previous verified artifacts; never silently regenerate them.
4. Rerun helper/training cells with `--resume`. Modules restore full state from Drive and skip completed stages. Starting a new branch resets optimizer/RNGs; resuming an interrupted branch restores them.
5. Rerun evaluation/plotting; cached identical checkpoint metrics are reused. Check Drive outputs before disconnecting.

Package/hardware changes may affect reproducibility; resolve and document them explicitly. Colab GPU availability is not guaranteed. Do not substitute full local CPU runs. See [Colab FAQ](https://research.google.com/colaboratory/faq.html).